<figure>
  <IMG SRC="https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Fachhochschule_Südwestfalen_20xx_logo.svg/320px-Fachhochschule_Südwestfalen_20xx_logo.svg.png" WIDTH=250 ALIGN="right">
</figure>

# Einführung Machine Learning
### Sommersemester 2026
Prof. Dr. Heiner Giefers

# Aufgabe zur Logistischen Regression

## Kreditscoring-Datensatz

Der Datensatz *Kreditscoring zur Klassifikation von Kreditnehmern. 2010. Open Data LMU.* ([https://doi.org/10.5282/ubm/data.23](https://doi.org/10.5282/ubm/data.23)) beinhaltet 1000 Datensätze, die Vergaben von Privatkrediten durch eine süddeutsche Großbank dokumentieren. Die Daten wurden ursprünglich für Forschung und Lehre bereitgestellt und bieten eine realitätsnahe Grundlage für Klassifikationsaufgaben im Bereich Kreditrisiko.

Die Spalten beschreiben verschiedene Merkmale, die sich drei übergeordneten Bereichen zuordnen lassen:

- die **persönliche Situation** der Kreditnehmer (z. B. Alter, Familienstand),
- die **wirtschaftliche Situation** (z. B. Einkommen, Höhe und Laufzeit des Darlehens, vorhandene Sicherheiten),
- die **rechtliche Stellung** (z. B. Art der Beschäftigung, laufende Verpflichtungen, Anzahl der bestehenden Kredite).

Beispiele für enthaltene Variablen:
- `laufzeit` – Laufzeit des beantragten Kredits (Monate),
- `hoehe` – Höhe des beantragten Kredits,
- `sparkont` – Kategorie des Sparguthabens (ordinal),
- `beszeit` – Beschäftigungsdauer in Jahren (ordinal),
- `rate` – Rückzahlungsrate bezogen auf Einkommen (prozentual),
- `famges` – Familienstand und Anzahl unterhaltsberechtigter Personen,
- `wohnzeit` – Dauer des aktuellen Wohnsitzes (Jahre),
- `buerge` – Bürgschaftsverhältnis (z. B. allein, Mitunterzeichner),
- `verm` – Vermögensverhältnisse (z. B. Immobilie, Lebensversicherung),
- `alter` – Alter der Antragsteller.

Die **Zielgröße** (auch *Target Variable* genannt) ist die Spalte `kredit`. Sie ist binär kodiert:
- `1` bedeutet, dass der entsprechende Kredit **ordnungsgemäß zurückgezahlt** wurde,
- `0` bedeutet, dass der Kredit **nicht vollständig oder gar nicht zurückgezahlt** wurde (Ausfall).

### Zielsetzung

Ziel des maschinellen Lernens in diesem Szenario ist es, ein System zu entwickeln, das für eine neue, potenzielle Kreditvergabe möglichst zuverlässig vorhersagen kann, ob der beantragte Kredit mit hoher Wahrscheinlichkeit zurückgezahlt werden wird oder nicht. Die Entscheidung stützt sich dabei auf Muster, die aus den vorhandenen historischen Daten gelernt werden.

Dieses Verfahren ist unter dem Begriff **Kreditscoring** bekannt. Es ist in der Praxis weit verbreitet und stellt ein zentrales Element in der Risikobewertung und Entscheidungsfindung von Banken und anderen Kreditinstituten dar. Statistische Modelle und maschinelle Lernverfahren helfen dabei, objektive, datengestützte Entscheidungen zu treffen und Kreditrisiken zu minimieren.

Mehr Informationen und eine vollständige Beschreibung aller Merkmale finden sich unter:  
[https://data.ub.uni-muenchen.de/23/1/DETAILS.html](https://data.ub.uni-muenchen.de/23/1/DETAILS.html)

Wir importieren zuerst die Pandas Bibliothek und laden den Datensatz `kredit.csv` in einen `DataFrame`.

In [ ]:
import pandas as pd
import os
import urllib.request

url = "https://github.com/fhswf/datasets/raw/main/kredit.csv"
dfile = "./kredit.csv"

if not os.path.isfile(dfile):
    urllib.request.urlretrieve(url, dfile)

In [ ]:
import pandas as pd
df = pd.read_csv("kredit.csv")
df.head(10)

Mit `df.info()` und `df.describe()` erhalten wir einige Informationen über den Datensatz.

In [ ]:
df.info()

In [ ]:
df.describe()

Wir teilen nun die kompletten Daten in einen Trainings- und einen Test-Datensatz auf.
Dazu kann die Methode `train_test_split()` aus dem Modul `sklearn.model_selection` verwendet werden.
Der Parameter `test_size` legt den Anteil des Daten im Test-Datensatz fest.
Die Aufteilung der Datenpunkte erfolgt zufällig.
Falls Sie immer die gleiche Aufteilung vornehmen wollen (damit die Ergebnisse vergleichbar sind) können Sie durch Festlegen des Parameters `random_state` erzwingen, dass immer die gleichen Folgen von Zufallszahlen erzeugt werden.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.3, random_state=0)

In [ ]:
df.corr().kredit.abs().sort_values()

Wenn Sie testen wollen, wie gut das Modell mit einer Auswahl der Merkmale funktioniert, können Sie die Spalten im Datensatz entsprechend einschränken.

In [ ]:
from sklearn.model_selection import train_test_split
features = ["sparkont","rate"]
X_train, X_test, y_train, y_test = train_test_split(df[features],df.iloc[:,0],test_size=0.3, random_state=0)

**Hinweis:** Wie bereits in früheren Praktika behandelt, ist es bei vielen Modellen wichtig, die Eingabedaten auf eine einheitliche Skala zu bringen.

Die logistische Regression ist empfindlich gegenüber unterschiedlich skalierten Features. Wir verwenden daher `StandardScaler`, um die numerischen Merkmale auf Mittelwert 0 und Standardabweichung 1 zu normieren.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Für die Modellbildung verwenden wir nun wieder `sklearn`, diesmal mit einem logistischen Regressionsmodell.

Die Modellparameter können über die Attribute `intercept_` und `coef_` abgerufen werden.
Üblicherweise interessieren den Programmierer diese Werte nicht.
die Schätzung für einen neuen Datenpunkt kann ja ganz einfach mit der Funktion `predict()` berechnet werden.
Für uns sind die Informationen allerdings interessant, wenn wir die Methode `fit()` händisch nachprogrammieren wollen und so die jeweiligen gelerneten Modellparameter miteinander vergleichen können.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train,y_train)

model.intercept_,  model.coef_[0]

Nachdem Wir das Modell mit den Trainingsdaten trainiert haben, verwenden wir den Testdatensatz um die Qualität des Modells zu bewerten.
Eine Vorhersagegenauigkeit von 66,6% bedeutet, dass für 2 von 3 Krediten korrekt vorhergesagt werden konnte, ob ein Kredit vom Bankkunden ordnungsgemäß zurückgezahlt wurde.

In [ ]:
import numpy as np
y_pred = model.predict(X_test)
acc_train = np.sum((y_pred==y_test)*1)/len(y_test)
print("Accuracy (Testdaten): %.2f%%" % (acc_train*100))

**Aufgabe:** **Verwenden nun alle Merkmale des Datensatzes in einem neuen Modell und berechnen Sie die *Classification Accuracy* für dieses Modell.**

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()

**Einordnung der Accuracy**

Das Modell erreicht auf den Testdaten eine Accuracy von ca. **75%**. Das bedeutet, dass etwa drei von vier Kreditentscheidungen korrekt vorhergesagt wurden – also ob ein Kredit zurückgezahlt wird (`kredit=1`) oder nicht (`kredit=0`).

**Ist das gut?**

Das hängt vom Anwendungsfall ab:

- In vielen klassischen Klassifikationsaufgaben (z. B. Bilderkennung) wäre eine Accuracy von 75 % eher mittelmäßig.
- Im **Kreditscoring** ist die Situation jedoch komplexer: Eine *falsche Entscheidung* kann entweder bedeuten, dass
  - **ein sicherer Kredit abgelehnt wird** (*false negative*) oder
  - **ein riskanter Kredit vergeben wird** (*false positive*).

Je nach Geschäftspriorität (z. B. Ausfallsvermeidung vs. Umsatzsteigerung) sind unterschiedliche Fehlertypen unterschiedlich kritisch. Daher ist die **reine Accuracy nicht ausreichend**, um ein Modell abschließend zu bewerten.

**Was fehlt?**

Für eine fundierte Bewertung benötigen wir zusätzliche Metriken wie:
- **Precision**: Wie zuverlässig sind die als „Zahlungsfähig“ klassifizierten Fälle?
- **Recall**: Wie viele der tatsächlichen Rückzahler hat das Modell erkannt?
- **F1-Score**: Kombination aus Precision und Recall.
- **Konfusionsmatrix**: Zeigt, welche Fehlertypen das Modell macht.

Diese Metriken werden wir in einem späteren Praktikum noch ausführlich behandeln.

**Fazit:**
Eine Accuracy von über 75 % ist ein **guter Startpunkt** für ein erstes Modell im Kreditbereich. In der Praxis würde man nun weiter analysieren, **wie sich das Modell verhält**, **welche Merkmale wichtig sind**, und wie man **Risiken und Chancen** balancieren kann.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError()